# Modeling
## March 2, 2026

# Data Dictionary

- id - unique ID for workout
- user_id - unique ID for athlete who did the workout
- sport_type - sport for the workout (~40 unique)
- sport_type_grouped - groups workouts into main groups
- speed_mph - miles / elapsed time in hours
- distance - distance in meters
- miles - distance in miles
- kilometers - distance in kilometers
- moving_time - seconds of active moving time (pauses for red light, water break, etc)
- elapsed_time - total seconds for entire workout
- moving_minutes - minutes of active moving time
- elapsed_minutes - minutes for entire workout
- moving_time_per - moving_minutes / elapsed_minutes
- total_elevation_gain - meters of climbing
- meters_per_km - avg meters of climbing per kilometer
- feet_per_mile - avg feet of climbing per mile (for the Americans lol)
- commute - boolean flag is user marked the activity as a commute (like when Oliver bikes to class)
- manual - flag for if the workout was generated by a tracking device or if user manually entered the details
- has_gear - boolean for if user indicated which shoes/bike they used for the workout
- suffer_score - Strava metric used to describe how tough the workout is; function of heart rate and total time
- kudos_count - how many "likes" the workout received on Strava
- device_name - name used to record the workout
- start_date - date of workout
- hour - hour of workout (start)
- day_part - morning vs afternoon vs evening vs night (start)
- month - month of workout
- dayofweek - day of week of workout
- is_weekend - boolean for if dayofweek == Saturday or Sunday
- is_northern_hemisphere - start_lat > 0
- num_turns - number of turns in the GPS trace
- turns_per_mile - num_turns / miles
- wobble - how wiggly vs straight the trace is (ignoring turns)
- sprawl - distance (in miles) from most northwest vs most southeast points in the trace
- is_winter - workout in Dec-Feb for northern hemisphere, or July-August for southern
- is_summer - workout in Dec-Feb for southern hemisphere, or July-August for northern

In [1]:
import pandas as pd

df = pd.read_csv("data/data_for_modeling.csv")

# create winter flag
df['is_winter'] = (
    ((df['is_northern_hemisphere'] == 1) & (df['month'].isin([12, 1, 2]))) |
    ((df['is_northern_hemisphere'] == 0) & (df['month'].isin([6, 7, 8])))
).astype(int)

# create summer flag
df['is_summer'] = (
    ((df['is_northern_hemisphere'] == 1) & (df['month'].isin([6, 7, 8]))) |
    ((df['is_northern_hemisphere'] == 0) & (df['month'].isin([12, 1, 2])))
).astype(int)

### Target variable

sport_type_grouped

### Features to remove

id / user_id / sport_type / start_date / device_name / suffer_score / is_northern_hemisphere

### Feature Selection

* miles <- distance/miles/kilometers
* moving_time <- moving_time/moving_minutes
* elapsed_time <- elapsed_time/elapsed_minutes
* feet_per_mile <- total_elevation_gain/meters_per_km

### Binary Features

commute / manual / has_gear / is_weekend / is_northern_hemisphere

### Categorical -> One-hot encoding

day_part

### GPS features -> combine into binary variable(has_gps)
⁠num_turns / turns_per_mile / wobble / sprawl



In [2]:
df['has_gps'] = df['num_turns'].notna().astype(int)

gps_cols = ['num_turns','turns_per_mile','wobble','sprawl']
df[gps_cols] = df[gps_cols].fillna(0)

df['has_gps'].value_counts()

has_gps
1    342829
0     55084
Name: count, dtype: int64

In [3]:
# Remove Anomalies
df = df[df['is_anomaly']==0].copy()

In [4]:
# Define Target and Features
y = df['sport_type_grouped']

final_feature_list = [
    # Core workout intensity / size
    "speed_mph",
    "miles",
    "moving_time",
    "elapsed_time",
    "moving_time_per",
    "feet_per_mile",

    # Route / GPS-shape features (+ indicator)
    "has_gps",
    "num_turns",
    "turns_per_mile",
    "wobble",
    "sprawl",

    # Time patterns
    "hour",
    "month",
    "dayofweek",
    "is_weekend",

    # Context flags
    "commute",
    "manual",
    "has_gear",
    "is_winter",
    "is_summer",

    # Light engagement signal (optional but usually safe)
    "kudos_count",

    # Categorical (we will one-hot encode later)
    "day_part",
]

X = df[final_feature_list].copy()


In [5]:
# Leakage Sanity Check
leak_cols = {"id", "user_id", "sport_type", "sport_type_grouped", "start_date", "device_name", "suffer_score", "is_anomaly"}
print("Leakage cols in X:", leak_cols.intersection(X.columns))

# Confirm no missing values remain after GPS filling
print(X.isna().sum().sort_values(ascending=False).head(10))


Leakage cols in X: set()
speed_mph      0
miles          0
kudos_count    0
is_summer      0
is_winter      0
has_gear       0
manual         0
commute        0
is_weekend     0
dayofweek      0
dtype: int64


In [6]:
# Encode categorical variables
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify catergorical + numeric columns
categorical_cols = ["day_part"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# Preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

In [7]:
# Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
sport_type_grouped
Ride       0.419841
Run        0.366082
Workout    0.093403
Walk       0.091773
Hike       0.028901
Name: proportion, dtype: float64

Test distribution:
sport_type_grouped
Ride       0.419838
Run        0.366083
Workout    0.093412
Walk       0.091766
Hike       0.028901
Name: proportion, dtype: float64


In [8]:
# Define Evaluation Metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [9]:
# Scaling
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Full pipeline
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("scaling", StandardScaler(with_mean=False)),  # required after one-hot
    ("classifier", LogisticRegression(max_iter=1000))
])

In [10]:
# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Metrics
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1:  {macro_f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix (pretty)
cm = confusion_matrix(y_test, y_pred, labels=model.named_steps["classifier"].classes_)
cm_df = pd.DataFrame(
    cm,
    index=model.named_steps["classifier"].classes_,
    columns=model.named_steps["classifier"].classes_
)
print("\nConfusion Matrix:")
print(cm_df)

Accuracy: 0.8579
Macro F1:  0.7414

Classification Report:
              precision    recall  f1-score   support

        Hike       0.63      0.32      0.42      2300
        Ride       0.93      0.89      0.91     33412
         Run       0.85      0.91      0.88     29134
        Walk       0.69      0.79      0.73      7303
     Workout       0.80      0.73      0.76      7434

    accuracy                           0.86     79583
   macro avg       0.78      0.73      0.74     79583
weighted avg       0.86      0.86      0.86     79583


Confusion Matrix:
         Hike   Ride    Run  Walk  Workout
Hike      726     41    211  1164      158
Ride       16  29861   2780   271      484
Run        98   1394  26552   685      405
Walk      250    101    928  5741      283
Workout    64    649    857   472     5392


# Option. Hist Gradient Boosting

In [15]:
#Step A) Pipeline
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,   # stops when no improvement on internal validation
)

model_hgb = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", hgb)
])


In [16]:
# Step B) Fit + Evaluate
model_hgb.fit(X_train, y_train)

y_pred = model_hgb.predict(X_test)



In [17]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9151074978324516
Macro F1: 0.8393560747289565

Classification Report:
               precision    recall  f1-score   support

        Hike       0.66      0.59      0.62      2300
        Ride       0.97      0.94      0.95     33412
         Run       0.92      0.95      0.93     29134
        Walk       0.79      0.88      0.83      7303
     Workout       0.89      0.83      0.86      7434

    accuracy                           0.92     79583
   macro avg       0.84      0.84      0.84     79583
weighted avg       0.92      0.92      0.92     79583


Confusion Matrix:
 [[ 1355    19    82   804    40]
 [   66 31257  1512   116   461]
 [  145   728 27632   471   158]
 [  424    44   323  6417    95]
 [   58   308   573   329  6166]]


In [ ]:
# HyperParameter Tuning
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist = {
    "classifier__learning_rate": [0.03, 0.05, 0.08, 0.1],
    "classifier__max_depth": [None, 3, 5, 8],
    "classifier__max_leaf_nodes": [15, 31, 63, 127],
    "classifier__min_samples_leaf": [10, 20, 50, 100],
    "classifier__l2_regularization": [0.0, 0.1, 1.0],
}

search = RandomizedSearchCV(
    estimator=model_hgb,
    param_distributions=param_dist,
    n_iter=20,                 # keep small first
    scoring="f1_macro",
    cv=3,
    random_state=42,
    n_jobs=-1,                 # parallelize CV (works here)
    verbose=2
)

search.fit(X_train, y_train)

print("Best Hist GB params:", search.best_params_)
print("Best CV macro F1:", search.best_score_)

best_model = search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("\nTEST Macro F1:", f1_score(y_test, y_pred_best, average="macro"))
print("\nClassification Report:\n", classification_report(y_test, y_pred_best))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END classifier__l2_regularization=1.0, classifier__learning_rate=0.08, classifier__max_depth=3, classifier__max_leaf_nodes=127, classifier__min_samples_leaf=10; total time=  13.5s
[CV] END classifier__l2_regularization=1.0, classifier__learning_rate=0.08, classifier__max_depth=3, classifier__max_leaf_nodes=127, classifier__min_samples_leaf=10; total time=  13.8s
[CV] END classifier__l2_regularization=1.0, classifier__learning_rate=0.08, classifier__max_depth=3, classifier__max_leaf_nodes=127, classifier__min_samples_leaf=10; total time=  13.9s
[CV] END classifier__l2_regularization=1.0, classifier__learning_rate=0.08, classifier__max_depth=8, classifier__max_leaf_nodes=15, classifier__min_samples_leaf=50; total time=  17.4s
[CV] END classifier__l2_regularization=1.0, classifier__learning_rate=0.08, classifier__max_depth=8, classifier__max_leaf_nodes=15, classifier__min_samples_leaf=50; total time=  17.4s
[CV] END classif

In [19]:
print("Best params:", search.best_params_)
print("Best CV macro F1:", search.best_score_)


Best params: {'classifier__min_samples_leaf': 10, 'classifier__max_leaf_nodes': 63, 'classifier__max_depth': 8, 'classifier__learning_rate': 0.1, 'classifier__l2_regularization': 1.0}
Best CV macro F1: 0.8543054505316438


In [20]:
# Refit the best model on the full training set

best_hgb = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    learning_rate=0.1,
    max_depth=8,
    max_leaf_nodes=63,
    min_samples_leaf=10,
    l2_regularization=1.0
)

best_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", best_hgb)
])

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

In [23]:
print("Tuned Hist GB Accuracy:", accuracy_score(y_test, y_pred))
print("Tuned Hist GB Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=best_model.named_steps["classifier"].classes_,
                     columns=best_model.named_steps["classifier"].classes_)
print("\nConfusion Matrix:\n", cm_df)

Tuned Hist GB Accuracy: 0.9263536182350502
Tuned Hist GB Macro F1: 0.8577731883269777

Classification Report:
               precision    recall  f1-score   support

        Hike       0.72      0.61      0.66      2300
        Ride       0.97      0.95      0.96     33412
         Run       0.93      0.96      0.94     29134
        Walk       0.81      0.89      0.85      7303
     Workout       0.90      0.86      0.88      7434

    accuracy                           0.93     79583
   macro avg       0.87      0.85      0.86     79583
weighted avg       0.93      0.93      0.93     79583


Confusion Matrix:
          Hike   Ride    Run  Walk  Workout
Hike     1404     19     84   749       44
Ride       17  31598   1296    90      411
Run       117    673  27841   365      138
Walk      376     44    285  6511       87
Workout    49    256    486   275     6368


Hyperparameter tuning of the HistGradientBoosting model improved overall performance from a macro F1 of 0.84 to 0.86 on the test set. Notably, minority class performance (Hike) improved substantially while maintaining strong performance on dominant classes such as Ride and Run. This indicates better class balance and generalization without sacrificing overall accuracy.

# Option2. Gradient Boosting + RandomizedSearchCV

In [26]:
# Pipeline
gb_random_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

In [27]:
# Random search space 
from sklearn.model_selection import StratifiedKFold

param_dist = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__max_depth": [2, 3, 4],           # depth of individual trees
    "classifier__min_samples_split": [2, 10, 30, 50],
    "classifier__min_samples_leaf": [1, 5, 10, 20],
    "classifier__subsample": [0.6, 0.8, 1.0],     # stochastic gradient boosting
    "classifier__max_features": [None, "sqrt", "log2"]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=gb_random_pipe,
    param_distributions=param_dist,
    n_iter=25,                    # start with 25; increase if runtime is OK
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)



In [28]:
search.fit(X_train, y_train)

print("Best GB with RandomSearch params:", search.best_params_)
print("Best CV macro F1:", search.best_score_)

Fitting 3 folds for each of 25 candidates, totalling 75 fits
[CV] END classifier__learning_rate=0.1, classifier__max_depth=2, classifier__max_features=sqrt, classifier__min_samples_leaf=10, classifier__min_samples_split=30, classifier__n_estimators=200, classifier__subsample=0.8; total time= 2.0min
[CV] END classifier__learning_rate=0.1, classifier__max_depth=2, classifier__max_features=sqrt, classifier__min_samples_leaf=10, classifier__min_samples_split=30, classifier__n_estimators=200, classifier__subsample=0.8; total time= 2.0min
[CV] END classifier__learning_rate=0.1, classifier__max_depth=2, classifier__max_features=sqrt, classifier__min_samples_leaf=10, classifier__min_samples_split=30, classifier__n_estimators=200, classifier__subsample=0.8; total time= 2.0min
[CV] END classifier__learning_rate=0.03, classifier__max_depth=2, classifier__max_features=log2, classifier__min_samples_leaf=5, classifier__min_samples_split=30, classifier__n_estimators=500, classifier__subsample=0.8; to

Exception ignored in: <function ResourceTracker.__del__ at 0x120065bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x103851bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x108079bc0>
Traceback (most recent call last

[CV] END classifier__learning_rate=0.03, classifier__max_depth=4, classifier__max_features=None, classifier__min_samples_leaf=1, classifier__min_samples_split=30, classifier__n_estimators=300, classifier__subsample=1.0; total time=30.4min
[CV] END classifier__learning_rate=0.03, classifier__max_depth=4, classifier__max_features=None, classifier__min_samples_leaf=1, classifier__min_samples_split=30, classifier__n_estimators=300, classifier__subsample=1.0; total time=30.4min
[CV] END classifier__learning_rate=0.03, classifier__max_depth=4, classifier__max_features=None, classifier__min_samples_leaf=1, classifier__min_samples_split=30, classifier__n_estimators=300, classifier__subsample=1.0; total time=30.4min


Exception ignored in: <function ResourceTracker.__del__ at 0x107281bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1037d9bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107951bc0>
Traceback (most recent call last

Best GB with RandomSearch params: {'classifier__subsample': 1.0, 'classifier__n_estimators': 500, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 5, 'classifier__max_features': 'log2', 'classifier__max_depth': 4, 'classifier__learning_rate': 0.1}
Best CV macro F1: 0.8465199100361351


In [29]:
best_gb_random_model = search.best_estimator_
y_gb_random_pred = best_model.predict(X_test)

print("\nGB Randomsearch Accuracy:", accuracy_score(y_test, y_pred))
print("GB Randomsearch Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("\nGB Randomsearch Classification Report:\n", classification_report(y_test, y_pred))
print("GB Randomsearch Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


GB Randomsearch Accuracy: 0.9263536182350502
GB Randomsearch Macro F1: 0.8577731883269777

GB Randomsearch Classification Report:
               precision    recall  f1-score   support

        Hike       0.72      0.61      0.66      2300
        Ride       0.97      0.95      0.96     33412
         Run       0.93      0.96      0.94     29134
        Walk       0.81      0.89      0.85      7303
     Workout       0.90      0.86      0.88      7434

    accuracy                           0.93     79583
   macro avg       0.87      0.85      0.86     79583
weighted avg       0.93      0.93      0.93     79583

GB Randomsearch Confusion Matrix:
 [[ 1404    19    84   749    44]
 [   17 31598  1296    90   411]
 [  117   673 27841   365   138]
 [  376    44   285  6511    87]
 [   49   256   486   275  6368]]


## Model Performance Comparison

| Model                         | Accuracy | Macro F1 | Minority Class (Hike F1) | Overall Assessment |
|--------------------------------|----------|----------|---------------------------|--------------------|
| Logistic Regression            | ~0.86    | ~0.74    | Low (~0.42 earlier)       | Underfits nonlinear patterns |
| HistGradientBoosting (Tuned)   | 0.926    | 0.858    | 0.66                      | Best balance & generalization |
| GradientBoosting (RandomSearch)| 0.926    | 0.858    | 0.66                      | Comparable to HistGB |

### Hist Gradient Boosting Classifier 
* Full RandomizedSearchCV tuning
* 500 estimators
* max_depth = 4
* learning_rate = 0.1
* subsample = 1.0
* max_features = log2

Rationale:

* Highest Macro F1
* Stable performance
* Computationally efficient
* Handles nonlinear interactions effectively


# Option3. XGBoost

In [30]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [31]:
from xgboost import XGBClassifier

In [38]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

xgb_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", xgb)
])

In [34]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_encoded = le.fit_transform(y)

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [39]:
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
print("XGB Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGB Macro F1:", f1_score(y_test, y_pred_xgb, average="macro"))
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

XGB Accuracy: 0.9258133018358192
XGB Macro F1: 0.8583732304876032
              precision    recall  f1-score   support

           0       0.72      0.62      0.67      2300
           1       0.97      0.95      0.96     33412
           2       0.93      0.96      0.94     29134
           3       0.82      0.89      0.85      7303
           4       0.90      0.85      0.88      7434

    accuracy                           0.93     79583
   macro avg       0.87      0.85      0.86     79583
weighted avg       0.93      0.93      0.93     79583

[[ 1430    20    82   732    36]
 [   15 31592  1290    88   427]
 [  110   689 27826   366   143]
 [  383    47   278  6497    98]
 [   49   265   502   284  6334]]


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

param_dist = {
    "classifier__n_estimators": [300, 500, 800],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__max_depth": [3, 4, 6, 8],
    "classifier__min_child_weight": [1, 5, 10],
    "classifier__subsample": [0.6, 0.8, 1.0],
    "classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "classifier__reg_lambda": [0.5, 1.0, 2.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search_xgb.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.03, classifier__max_depth=4, classifier__min_child_weight=5, classifier__n_estimators=500, classifier__reg_lambda=2.0, classifier__subsample=0.6; total time=  47.9s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.03, classifier__max_depth=4, classifier__min_child_weight=5, classifier__n_estimators=500, classifier__reg_lambda=2.0, classifier__subsample=0.6; total time=  48.0s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.03, classifier__max_depth=4, classifier__min_child_weight=5, classifier__n_estimators=500, classifier__reg_lambda=2.0, classifier__subsample=0.6; total time=  48.9s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.03, classifier__max_depth=4, classifier__min_child_weight=10, classifier__n_estimators=800, classifier__reg_lambda=1.0, classifier__subsample=1.0; total time= 1

RandomizedSearchCV(cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('cat',
                                                                               OneHotEncoder(drop='first'),
                                                                               ['day_part']),
                                                                              ('num',
                                                                               'passthrough',
                                                                               ['speed_mph',
                                                                                'miles',
                                                                                'moving_time',
                                                                                'elapsed_time',
                                                                                'moving_time_per',
                                                                                'feet_per_mile',
                                                                                'has_gps',
                                                                                'num_turns',
                                                                                'turns_per_mile',
                                                                                'wo...
                   n_iter=20, n_jobs=-1,
                   param_distributions={'classifier__colsample_bytree': [0.6,
                                                                         0.8,
                                                                         1.0],
                                        'classifier__learning_rate': [0.03,
                                                                      0.05,
                                                                      0.1],
                                        'classifier__max_depth': [3, 4, 6, 8],
                                        'classifier__min_child_weight': [1, 5,
                                                                         10],
                                        'classifier__n_estimators': [300, 500,
                                                                     800],
                                        'classifier__reg_lambda': [0.5, 1.0,
                                                                   2.0],
                                        'classifier__subsample': [0.6, 0.8,
                                                                  1.0]},
                   random_state=42, scoring='f1_macro', verbose=2)

Exception ignored in: <function ResourceTracker.__del__ at 0x110065bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x112165bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x108cc5bc0>
Traceback (most recent call last

In [41]:
best_xgb = search_xgb.best_estimator_

y_pred_best = best_xgb.predict(X_test)

print("Best XGB params:", search_xgb.best_params_)
print("XGB TEST Accuracy:", accuracy_score(y_test, y_pred_best))
print("XGB TEST Macro F1:", f1_score(y_test, y_pred_best, average="macro"))
print(classification_report(y_test, y_pred_best))

Best XGB params: {'classifier__subsample': 0.6, 'classifier__reg_lambda': 0.5, 'classifier__n_estimators': 800, 'classifier__min_child_weight': 1, 'classifier__max_depth': 8, 'classifier__learning_rate': 0.1, 'classifier__colsample_bytree': 1.0}
XGB TEST Accuracy: 0.9344206677305454
XGB TEST Macro F1: 0.8696656655330323
              precision    recall  f1-score   support

           0       0.73      0.63      0.68      2300
           1       0.97      0.95      0.96     33412
           2       0.94      0.96      0.95     29134
           3       0.83      0.90      0.86      7303
           4       0.92      0.88      0.90      7434

    accuracy                           0.93     79583
   macro avg       0.88      0.86      0.87     79583
weighted avg       0.93      0.93      0.93     79583



## Model Comparison

| Model | Accuracy | Macro F1 | Notes |
|------|------|------|------|
| Logistic Regression | ~0.86 | ~0.74 | Linear baseline |
| GradientBoosting | 0.926 | 0.858 | Nonlinear ensemble |
| HistGradientBoosting | 0.926 | 0.858 | Faster boosting |
| **XGBoost (tuned)** | **0.934** | **0.870** | Best overall performance |